In [1]:
import os
print(os.getcwd())

/home/amahaj56/NLP_Project/NEW FOLDER


In [1]:
# FOR JOB

from vllm import LLM, SamplingParams
model_name = "allenai/OLMo-2-1124-7B-Instruct"
llm = LLM(
    model=model_name,
    dtype="float16",
    gpu_memory_utilization=0.85  # safe for single GPU
)
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024
)

import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]
mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict.pop('en', None)
mapping_dict.pop('fr', None)
mapping_dict.pop('es', None)
mapping_dict.pop('zh', None)
mapping_dict.pop('hi', None)
mapping_dict.pop('ko', None)
print(len(mapping_dict))

from datasets import *
ds = load_dataset("json", data_files="mgsm_multicolumn_multilang.jsonl", split="train")

ref = load_dataset("juletxara/mgsm", "en", split="test")
references = [ex["answer_number"] for ex in ref]

import re
def extract_final_answer(text: str):
    # Regex matches numbers with optional commas, decimals, and negatives
    numbers = re.findall(r'-?\d[\d,]*\.?\d*', text)
    if not numbers:
        return None
    return numbers[-1]  # return the last number

def compute_accuracy(predictions, gold_answers):
    correct = 0
    total = len(predictions)
    for pred_text, gold in zip(predictions, gold_answers):
        pred_num = extract_final_answer(pred_text)
        if pred_num is not None:
            if pred_num.endswith('.'):
                pred_num = pred_num[:-1]
            pred_num_norm = pred_num.replace(",", "")
        else:
            pred_num_norm = None
        gold_norm = str(gold)
        
        if pred_num_norm == gold_norm:
            correct += 1
    accuracy = correct / total
    return accuracy


# translations
import json
with open("translations.json", "r", encoding="utf-8") as f:
    translations = json.load(f)
multi_lang_prompt = {}

for lang_code, parts in translations.items():
    # Compose the Step-by-Step example by joining steps + answer (mirrors the 'en' example)
    example_steps = (parts.get("steps", "").strip() + " " + parts.get("answer", "").strip()).strip()
    # Build the prompt template with a {} placeholder for the new question
    prompt = f"""{parts.get("instruction", "").strip()}
    Question: {parts.get("question", "").strip()}
    Step-by-Step Answer: {example_steps}
    Final Answer: {parts.get("answer", "").strip()}
    Question: {{}}
    Step-by-Step Answer:  
    Final Answer:"""
    multi_lang_prompt[lang_code] = prompt


results = {}
for lang in list(mapping_dict.keys())[19:]:
    print(lang)
    prompts = [multi_lang_prompt[lang].format(ex[f"question_{lang}"]) for ex in ds]
    batch_size = 32
    all_outputs = []
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i + batch_size]
        outputs = llm.generate(batch_prompts, sampling_params)
        for out in outputs:
            all_outputs.append(out.outputs[0].text.strip())
    results[lang] = compute_accuracy(all_outputs, references)
    print(results)

print("="*60)
print(results)

INFO 11-07 05:59:24 [__init__.py:216] Automatically detected platform cuda.
INFO 11-07 05:59:25 [utils.py:328] non-default args: {'dtype': 'float16', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'allenai/OLMo-2-1124-7B-Instruct'}
INFO 11-07 05:59:34 [__init__.py:742] Resolved architecture: Olmo2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 11-07 05:59:34 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 11-07 05:59:34 [__init__.py:1815] Using max model len 4096
INFO 11-07 05:59:34 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1971624) INFO 11-07 05:59:35 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=1971624) INFO 11-07 05:59:35 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='allenai/OLMo-2-1124-7B-Instruct', speculative_config=None, tokenizer='allenai/OLMo-2-1124-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', di

[W1107 05:59:38.954472623 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=1971624) INFO 11-07 05:59:38 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=1971624) INFO 11-07 05:59:38 [cuda.py:362] Using Flash Attention backend on V1 engine.
(EngineCore_DP0 pid=1971624) INFO 11-07 05:59:39 [weight_utils.py:348] Using model weights format ['*.safetensors']


...

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

ERROR 11-07 06:25:16 [core_client.py:564] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


KeyboardInterrupt: 

In [2]:
mgsm1 = {'als': 0.048, 'arz': 0.112, 'ast': 0.132, 'azb': 0.032, 'ceb': 0.052, 'ckb': 0.016, 'ilo': 0.04, 'lmo': 0.144, 'mai': 0.016, 'min': 0.064, 'scn': 0.132, 'vec': 0.176, 'war': 0.044, 'yue': 0.1, 'af': 0.104, 'am': 0.032, 'as': 0.0, 'ba': 0.024, 'be': 0.048}
mgsm2 = {'bn': 0.02, 'bo': 0.004, 'bs': 0.084, 'bg': 0.096, 'ca': 0.144, 'cs': 0.096, 'cy': 0.06, 'da': 0.12, 'de': 0.128, 'el': 0.092, 'eo': 0.056, 'et': 0.056, 'eu': 0.044, 'fi': 0.076, 'gd': 0.06, 'ga': 0.056, 'gl': 0.072, 'gn': 0.036, 'gu': 0.072, 'ht': 0.052, 'he': 0.124, 'hr': 0.1, 'hu': 0.032, 'hy': 0.032, 'id': 0.156, 'is': 0.068, 'it': 0.196, 'jv': 0.08, 'ja': 0.096, 'kn': 0.048, 'ka': 0.02, 'kk': 0.032, 'km': 0.02}
mgsm3 = {'yi': 0.028, 'sw': 0.044, 'yo': 0.012, 'vi': 0.116, 'ur': 0.112, 'uk': 0.136, 'ug': 0.02, 'tr': 0.116, 'tk': 0.044, 'th': 0.064, 'tl': 0.12, 'tg': 0.044, 'te': 0.096, 'tt': 0.02, 'ta': 0.052, 'sv': 0.128, 'su': 0.08, 'sr': 0.08, 'sc': 0.076, 'so': 0.028, 'sd': 0.052, 'sl': 0.06, 'sk': 0.056, 'si': 0.036, 'sa': 0.008, 'ru': 0.196, 'ro': 0.168, 'pt': 0.132, 'pl': 0.096, 'pa': 0.068, 'oc': 0.124, 'nn': 0.1, 'nl': 0.08, 'my': 0.0, 'mt': 0.032, 'mk': 0.06, 'mr': 0.048, 'ml': 0.052, 'lb': 0.032, 'lt': 0.068, 'li': 0.084, 'lo': 0.028, 'ky': 0.036}


In [3]:
len(mgsm1) + len(mgsm2) + len(mgsm3)

95